In [119]:
import json
import sys

In [120]:
time_point = ["E12.5","E13.5","E14.5","E16.5","E18.5","P1","P5","P56"]
tree_edge_fn = "../results/tree_results/tree_edge.txt"
edge_prob_fn = "../results/tree_results/edge_prob.txt"
celltype_groups_fn = "../test_data/celltype_groups.txt"

In [122]:
time_point = ["root"] + time_point
time_point_n = len(time_point)
time_point_id = {}
for i in range(0,time_point_n):
    time_point_id[time_point[i]] = i


### read edge info 
edge = {}; node_all = []; node_each = {}; main_edge = set()

file = open(tree_edge_fn)
for line in file:
    l = line.rstrip().split('\t')

    if l[0] not in node_all:
        node_all.append(l[0])
    if l[1] not in node_all:
        node_all.append(l[1])

    edge[l[0]] = edge.get(l[0], [])
    edge[l[0]].append(l[1])

    num = time_point_id[l[0].split(':')[0]]
    node_each[num] = node_each.get(num, [])
    if l[0] not in node_each[num]:
        node_each[num].append(l[0])

    main_edge.add((l[0],l[1]))

### read edge weight file
main_weight = {}; extra_weight = {}
file = open(edge_prob_fn)
for line in file:
    l = line.rstrip().split('\t')
    if float(l[2])>0.8:
        xx = "specific"
    elif float(l[2])>0.2:
        xx = "additional"
    if (l[0],l[1]) in main_edge:
        if float(l[2]) > 0.2:
            main_weight[l[1]] = float(l[2])
        else:
            main_weight[l[1]] = 0
    else:
        if float(l[2]) > 0.2:
            extra_weight[(l[0],l[1])] = float(l[2])

In [123]:
node_all

['root:root',
 'E12.5:CE_SE',
 'E12.5:End',
 'E12.5:Eryt',
 'E12.5:Germ',
 'E12.5:Immune',
 'E12.5:MM',
 'E12.5:MT',
 'E12.5:pre-Gran',
 'E12.5:SLC',
 'E12.5:SP',
 'E13.5:CE_SE',
 'E13.5:End',
 'E13.5:Eryt',
 'E13.5:Germ',
 'E13.5:Immune',
 'E13.5:SP',
 'E13.5:pre-Gran',
 'E13.5:SLC',
 'E14.5:CE_SE',
 'E14.5:End',
 'E14.5:Eryt',
 'E14.5:Germ',
 'E14.5:Immune',
 'E14.5:pre-Gran',
 'E14.5:SLC',
 'E14.5:SP',
 'E16.5:CE_SE',
 'E16.5:End',
 'E16.5:Eryt',
 'E16.5:Germ',
 'E16.5:Immune',
 'E16.5:pre-Gran',
 'E16.5:SLC',
 'E16.5:PV',
 'E16.5:SP',
 'E18.5:CE_SE',
 'E18.5:End',
 'E18.5:Eryt',
 'E18.5:Immune',
 'E18.5:Germ',
 'E18.5:pre-Gran',
 'E18.5:SLC',
 'E18.5:SP',
 'P1:CE_SE',
 'P1:End',
 'P1:Eryt',
 'P1:Germ',
 'P1:Immune',
 'P1:pre-Gran',
 'P1:SLC',
 'P1:PV',
 'P1:SP',
 'P5:CE_SE',
 'P5:End',
 'P5:Eryt',
 'P5:Germ',
 'P5:Immune',
 'P5:Gran',
 'P5:PV',
 'P5:SLC',
 'P5:SM',
 'P5:SP',
 'P56:CE_SE',
 'P56:End',
 'P56:Eryt',
 'P56:Gran',
 'P56:Stero',
 'P56:Immune',
 'P56:Luteal',
 'P56:SM',
 

In [124]:
extra_weight

{}

In [125]:
file = open(celltype_groups_fn)
i = 1
coor = {}
node_group = {}
color_map = {}
for line in file:
    l = line.rstrip().split("\t")
    node_group[l[0]] = int(l[1])
    coor[l[0]] = i
    color_map[int(l[1])] = l[2]
    i += 1
file.close()

In [126]:
len(node_all)

72

In [127]:
### create the info for all the node
dat = {}

for i in node_all:

    dat[i] = {'name':i}
    if i in edge:
        dat[i]['children'] = []

    if i in main_weight:
        dat[i]['edge_weight'] = main_weight[i]
    else:
        print(i)

    ### 10. coors of x (used to create the plot, the cell type order)
    if i.split(':')[1] in coor:
        dat[i]['fx'] = str(coor[i.split(':')[1]])
    else:
        print(i)


    if i.split(':')[1] in node_group:
        dat[i]['node_group'] = color_map[node_group[i.split(':')[1]]]
    else:
        print(i)

root:root
root:root
root:root


In [128]:
### add extra node first
for i in extra_weight:
    tmp = {'name': i[1], 'edge_weight': extra_weight[i], 'fx': str(coor[i[1].split(':')[1]]), 'node_group': color_map[node_group[i[1].split(':')[1]]]}
    dat[i[0]]['children'] = dat[i[0]].get('children',[])
    dat[i[0]]['children'].append(tmp)
    print(dat[i[0]]['children'])

In [129]:
extra_weight

{}

In [130]:
### connect each time point
for i in range(time_point_n-2, 0, -1):
    for j in node_each[i]:
        for k in edge[j]:
            dat[j]['children'].append(dat[k])

In [131]:
dat

{'root:root': {'name': 'root:root', 'children': []},
 'E12.5:CE_SE': {'name': 'E12.5:CE_SE',
  'children': [{'name': 'E13.5:CE_SE',
    'children': [{'name': 'E14.5:CE_SE',
      'children': [{'name': 'E16.5:CE_SE',
        'children': [{'name': 'E18.5:CE_SE',
          'children': [{'name': 'P1:CE_SE',
            'children': [{'name': 'P5:CE_SE',
              'children': [{'name': 'P56:CE_SE',
                'edge_weight': 0.986666666666667,
                'fx': '54',
                'node_group': '#bb5838'}],
              'edge_weight': 0.959748427672956,
              'fx': '54',
              'node_group': '#bb5838'}],
            'edge_weight': 0.913286713286713,
            'fx': '54',
            'node_group': '#bb5838'}],
          'edge_weight': 0.371717171717172,
          'fx': '54',
          'node_group': '#bb5838'}],
        'edge_weight': 0.951250086811584,
        'fx': '54',
        'node_group': '#bb5838'}],
      'edge_weight': 0.933065568724233,
      'fx': '54

In [133]:
for k in edge['root:root']:
    dat['root:root']['children'].append(dat[k])
dat_json = dat['root:root']


with open("../results/tree_results/tree.json", 'w') as json_file:
    json.dump(dat_json, json_file)